# SEC Form 13F-CRSP holdings audit

This notebook audits the analysis-ready holdings panel produced by `scripts/07_build_sec_13f_crsp_holdings.py`.

It checks:

1. the integrity of manager-quarter-PERMNO holdings;
2. reconciliation of holdings, portfolio totals, and portfolio weights;
3. sample retention through exact CRSP matching and the common-equity filter; and
4. the dimensions of the final holdings sample.

The notebook reads the saved outputs only and does not rebuild them.

## 1. Setup and inputs

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq


project_root = Path.cwd().resolve()

while not (project_root / "config" / "paths.yaml").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not locate the project root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.configuration import get_paths_config


paths = get_paths_config()
output_directory = paths.interim / "sec_13f_crsp"
holdings_path = output_directory / "manager_security_holdings.parquet"
totals_path = output_directory / "manager_portfolio_totals.parquet"
audit_path = output_directory / "sample_construction_audit.parquet"

for path in (holdings_path, totals_path, audit_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required Script 07 output not found: {path}")

inventory = pd.DataFrame(
    {
        "table": [
            "manager_security_holdings",
            "manager_portfolio_totals",
            "sample_construction_audit",
        ],
        "rows": [
            pq.ParquetFile(holdings_path).metadata.num_rows,
            pq.ParquetFile(totals_path).metadata.num_rows,
            pq.ParquetFile(audit_path).metadata.num_rows,
        ],
    }
)

connection = duckdb.connect(database=":memory:")
connection.read_parquet(str(holdings_path)).create_view("holdings")
connection.read_parquet(str(totals_path)).create_view("portfolio_totals")
sample_audit = pd.read_parquet(audit_path)

display(inventory.style.format({"rows": "{:,}"}))

,table,rows
0,manager_security_holdings,"40,413,958"
1,manager_portfolio_totals,"251,386"
2,sample_construction_audit,156


## 2. Holdings integrity

Each row should identify one eligible U.S. common-equity position for one manager, quarter, and CRSP PERMNO. Values and portfolio weights must be positive, and the CRSP classifications must satisfy the fixed eligibility rule.

In [2]:
holdings_summary = connection.execute(
    '''
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (CIK, report_period)) AS manager_quarters,
        COUNT(DISTINCT CIK) AS managers,
        COUNT(DISTINCT permno) AS permnos,
        COUNT(DISTINCT report_period) AS quarters,
        MIN(report_period) AS first_quarter,
        MAX(report_period) AS last_quarter
    FROM holdings
    '''
).fetchdf()

holdings_checks = connection.execute(
    '''
    SELECT
        COUNT(*) - COUNT(DISTINCT (CIK, report_period, permno))
            AS duplicate_keys,
        COUNT(*) FILTER (
            WHERE CIK IS NULL OR report_period IS NULL OR permno IS NULL
        ) AS missing_key_values,
        COUNT(*) FILTER (WHERE information_date < report_period)
            AS information_dates_before_report,
        COUNT(*) FILTER (WHERE mapped_cusip_count < 1)
            AS invalid_mapped_cusip_counts,
        COUNT(*) FILTER (
            WHERE position_value_usd <= 0 OR portfolio_value_usd <= 0
        ) AS nonpositive_values,
        COUNT(*) FILTER (
            WHERE portfolio_weight <= 0 OR portfolio_weight > 1
               OR NOT isfinite(portfolio_weight)
        ) AS invalid_portfolio_weights,
        COUNT(*) FILTER (
            WHERE securitytype != 'EQTY'
               OR securitysubtype != 'COM'
               OR sharetype != 'NS'
               OR usincflg != 'Y'
               OR primaryexch NOT IN ('N', 'A', 'Q')
        ) AS ineligible_securities
    FROM holdings
    '''
).fetchdf().T.reset_index()
holdings_checks.columns = ["check", "violations"]

display(holdings_summary.style.format({
    "rows": "{:,}",
    "manager_quarters": "{:,}",
    "managers": "{:,}",
    "permnos": "{:,}",
    "quarters": "{:,}",
}))
display(holdings_checks.style.format({"violations": "{:,}"}))

print()
print(f"Total holdings-integrity violations: {holdings_checks['violations'].sum():,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,manager_quarters,managers,permnos,quarters,first_quarter,last_quarter
0,"40,413,958","251,386","12,211","7,232",52,2013-06-30 00:00:00,2026-03-31 00:00:00


,check,violations
0,duplicate_keys,0
1,missing_key_values,0
2,information_dates_before_report,0
3,invalid_mapped_cusip_counts,0
4,nonpositive_values,0
5,invalid_portfolio_weights,0
6,ineligible_securities,0



Total holdings-integrity violations: 0


## 3. Portfolio reconciliation

The holdings are independently aggregated by manager-quarter and compared with the saved portfolio totals. Within each portfolio, weights should sum to one apart from negligible floating-point error.

In [3]:
connection.execute(
    '''
    CREATE TEMP TABLE holdings_by_portfolio AS
    SELECT
        CIK,
        PERIODOFREPORT,
        report_period,
        MIN(information_date) AS minimum_information_date,
        MAX(information_date) AS maximum_information_date,
        SUM(position_value_usd) AS holdings_value,
        MIN(portfolio_value_usd) AS minimum_stored_value,
        MAX(portfolio_value_usd) AS maximum_stored_value,
        COUNT(*) AS security_count,
        SUM(raw_position_rows) AS underlying_position_rows,
        SUM(portfolio_weight) AS weight_sum
    FROM holdings
    GROUP BY CIK, PERIODOFREPORT, report_period
    '''
)

reconciliation = connection.execute(
    '''
    SELECT
        COUNT(*) FILTER (WHERE h.CIK IS NULL) AS totals_without_holdings,
        COUNT(*) FILTER (WHERE t.CIK IS NULL) AS holdings_without_totals,
        COUNT(*) FILTER (
            WHERE h.minimum_information_date != h.maximum_information_date
               OR h.minimum_information_date != t.information_date
        ) AS information_date_mismatches,
        COUNT(*) FILTER (
            WHERE h.holdings_value != t.portfolio_value_usd
               OR h.minimum_stored_value != t.portfolio_value_usd
               OR h.maximum_stored_value != t.portfolio_value_usd
        ) AS portfolio_value_mismatches,
        COUNT(*) FILTER (WHERE h.security_count != t.security_count)
            AS security_count_mismatches,
        COUNT(*) FILTER (
            WHERE h.underlying_position_rows != t.underlying_position_rows
        ) AS source_row_mismatches,
        COUNT(*) FILTER (WHERE abs(h.weight_sum - 1) > 1e-9)
            AS invalid_weight_sums,
        MAX(abs(h.weight_sum - 1)) AS maximum_weight_sum_error
    FROM holdings_by_portfolio AS h
    FULL JOIN portfolio_totals AS t
        USING (CIK, PERIODOFREPORT, report_period)
    '''
).fetchdf()

reconciliation_checks = (
    reconciliation.drop(columns="maximum_weight_sum_error")
    .T.reset_index()
)
reconciliation_checks.columns = ["check", "violations"]

portfolio_distribution = connection.execute(
    '''
    SELECT
        MIN(security_count) AS minimum,
        MEDIAN(security_count) AS median,
        AVG(security_count) AS mean,
        MAX(security_count) AS maximum
    FROM portfolio_totals
    '''
).fetchdf()

display(reconciliation_checks.style.format({"violations": "{:,}"}))
print()
print(
    "Maximum absolute portfolio-weight sum error: "
    f"{reconciliation.at[0, 'maximum_weight_sum_error']:.3e}"
)
print()
print("Eligible securities per manager-quarter")
display(portfolio_distribution.style.format({
    "minimum": "{:,.0f}",
    "median": "{:,.0f}",
    "mean": "{:,.1f}",
    "maximum": "{:,.0f}",
}))

,check,violations
0,totals_without_holdings,0
1,holdings_without_totals,0
2,information_date_mismatches,0
3,portfolio_value_mismatches,0
4,security_count_mismatches,0
5,source_row_mismatches,0
6,invalid_weight_sums,0



Maximum absolute portfolio-weight sum error: 2.109e-15

Eligible securities per manager-quarter


,minimum,median,mean,maximum
0,1,57,160.8,"4,174"


## 4. Sample construction

The saved audit records three sequential stages: all Form 13F positions, exact CRSP matches, and eligible U.S. common equities. Position and manager retention are the main sample measures. Value retention is descriptive because unresolved historical SEC value-unit cases remain in the source data.

In [4]:
stage_order = [
    "FORM_13F_POSITIONS",
    "EXACT_CRSP_MATCHES",
    "ELIGIBLE_US_COMMON_EQUITIES",
]
sample_audit["stage"] = pd.Categorical(
    sample_audit["stage"], categories=stage_order, ordered=True
)
sample_audit = sample_audit.sort_values(["report_period", "stage"])

stage_totals = (
    sample_audit.groupby("stage", observed=True)[
        ["manager_positions", "manager_periods", "position_value_usd"]
    ]
    .sum()
    .reindex(stage_order)
    .reset_index()
)
baseline = stage_totals.iloc[0]
stage_totals["position_retention_percent"] = (
    100 * stage_totals["manager_positions"] / baseline["manager_positions"]
)
stage_totals["manager_retention_percent"] = (
    100 * stage_totals["manager_periods"] / baseline["manager_periods"]
)
stage_totals["value_retention_percent"] = (
    100 * stage_totals["position_value_usd"] / baseline["position_value_usd"]
)

eligible_quarters = sample_audit.loc[
    sample_audit["stage"].eq("ELIGIBLE_US_COMMON_EQUITIES")
]
quarterly_retention = (
    eligible_quarters[
        ["position_retention_percent", "manager_retention_percent"]
    ]
    .agg(["min", "median", "max"])
    .T.reset_index(names="measure")
)

audit_checks = pd.DataFrame(
    {
        "check": [
            "Duplicate quarter-stage rows",
            "Quarters without all three stages",
            "Quarters with increasing position counts",
            "Quarters with increasing manager counts",
        ],
        "violations": [
            sample_audit.duplicated(["report_period", "stage"]).sum(),
            sample_audit.groupby("report_period")["stage"].nunique().ne(3).sum(),
            sample_audit.groupby("report_period")["manager_positions"]
                .apply(lambda values: not values.is_monotonic_decreasing).sum(),
            sample_audit.groupby("report_period")["manager_periods"]
                .apply(lambda values: not values.is_monotonic_decreasing).sum(),
        ],
    }
)

display(stage_totals.style.format({
    "manager_positions": "{:,}",
    "manager_periods": "{:,}",
    "position_value_usd": "${:,.0f}",
    "position_retention_percent": "{:.2f}",
    "manager_retention_percent": "{:.2f}",
    "value_retention_percent": "{:.2f}",
}))
display(quarterly_retention.style.format({"min": "{:.2f}", "median": "{:.2f}", "max": "{:.2f}"}))
display(audit_checks.style.format({"violations": "{:,}"}))

,stage,manager_positions,manager_periods,position_value_usd,position_retention_percent,manager_retention_percent,value_retention_percent
0,FORM_13F_POSITIONS,"60,766,163","260,473","$2,885,297,927,045,115",100.00,100.00,100.00
1,EXACT_CRSP_MATCHES,"59,224,806","259,983","$2,683,175,826,370,207",97.46,99.81,92.99
2,ELIGIBLE_US_COMMON_EQUITIES,"40,415,180","251,386","$2,047,547,933,630,267",66.51,96.51,70.96


,measure,min,median,max
0,position_retention_percent,60.20,68.20,73.74
1,manager_retention_percent,95.66,96.28,97.06


,check,violations
0,Duplicate quarter-stage rows,0
1,Quarters without all three stages,0
2,Quarters with increasing position counts,0
3,Quarters with increasing manager counts,0


## 5. Coverage over time

The final table reports how the number of eligible manager portfolios and CRSP securities evolves across quarters.

In [5]:
quarterly_sample = connection.execute(
    '''
    SELECT
        report_period,
        COUNT(DISTINCT CIK) AS managers,
        COUNT(DISTINCT permno) AS permnos,
        COUNT(*) AS manager_security_holdings
    FROM holdings
    GROUP BY report_period
    ORDER BY report_period
    '''
).fetchdf()

display(quarterly_sample.head())
display(quarterly_sample.tail())

print()
print(
    "Eligible managers per quarter: "
    f"{quarterly_sample['managers'].min():,} to "
    f"{quarterly_sample['managers'].max():,}"
)
print(
    "Eligible PERMNOs per quarter: "
    f"{quarterly_sample['permnos'].min():,} to "
    f"{quarterly_sample['permnos'].max():,}"
)

connection.close()

,report_period,managers,permnos,manager_security_holdings
0,2013-06-30,2674,3786,478717
1,2013-09-30,2830,3804,523107
2,2013-12-31,3133,3820,551179
3,2014-03-31,3188,3853,570002
4,2014-06-30,3153,3900,580793


,report_period,managers,permnos,manager_security_holdings
47,2025-03-31,7176,3926,1103065
48,2025-06-30,7157,3881,1120638
49,2025-09-30,7135,3851,1129733
50,2025-12-31,7656,3823,1184810
51,2026-03-31,7823,3799,1222964



Eligible managers per quarter: 2,674 to 7,823
Eligible PERMNOs per quarter: 3,752 to 4,598


## 6. Interpretation

- **The holdings keys and eligibility rules are valid.** The final table contains 40,413,958 manager-quarter-PERMNO holdings across 251,386 manager-quarters, 12,211 managers, 7,232 PERMNOs, and 52 quarters. All integrity checks return zero violations.
- **Holdings and portfolio totals reconcile exactly.** Values, security counts, underlying source-row counts, and information dates agree for every manager-quarter. Portfolio weights sum to one with a maximum floating-point error of only `2.109e-15`.
- **The reduction in observations is transparent.** Exact CRSP matching retains 97.46% of Form 13F manager-position rows and 99.81% of manager-quarters. The U.S. common-equity screen retains 66.51% of positions and 96.51% of manager-quarters. The filter therefore removes many ineligible instruments while preserving nearly all manager portfolios.
- **Aggregation behaves as intended.** The 40,415,180 eligible CUSIP-level positions become 40,413,958 holdings because multiple eligible CUSIPs can map to the same manager-quarter-PERMNO. The saved `mapped_cusip_count` records these consolidations.
- **Portfolio sizes are heterogeneous.** Eligible portfolios contain a median of 57 securities and a mean of 160.8, ranging from 1 to 4,174 securities. This variation is retained rather than imposing an additional portfolio-size filter.

The Script 07 outputs are internally consistent and suitable as the holdings input to the ownership-network construction. Reported-value retention remains descriptive because it inherits the unresolved SEC value-unit cases documented in Notebook 03.